### V0.4

* 1) age      :int64	    顧客の年齢。 
* 2) job      :object	職業（admin, blue-collar, technician など）。　
* 3) marital  :object	既婚・未婚などの婚姻状況。　
* 4) education:object	学歴（primary, secondary, tertiary など）
                    ※primary: 初等教育（小学校など）
                     secondary: 中等教育（中学校・高校など）
                     tertiary: 高等教育（大学・大学院など）
                     unknown: 不明
* 5) default  :object	債務不履行があるかどうか（yes / no）　過去にローンやクレジットカードの支払いが滞ったことがあるか
* 6) balance  :int64	    年間の平均残高。
* 7) housing  :object	住宅ローンがあるかどうか（yes / no）。
* 8) loan     :object	個人ローンがあるかどうか（yes / no）
* 9) contact  :object	連絡手段（cellular（スマホ）, telephone （固定）など）。
* 10) day      :int64	    最終接触日の「日」。
* 11) month    :object	最終接触日の「月」。
* ※　duration削除 
* 13) campaign :int64     このキャンペーン（勧誘）期間中の接触回数。
* 14) pdays    :int64	    前回のキャンペーン（勧誘）接触から経過した日数（-1は未接触）。
* 15) previous :int64	    このキャンペーン（勧誘）以前の接触回数。
* 16) poutcome :object	前回のキャンペーン（勧誘）の成果（success(契約してくれた), failure(断られた) など）。
<br>
<br>
新しいカラム<br>

* 17) total_loan_count：int64　housingとloanの「yes」の合計数（負債の重さ）。
* 18) balance_per_age：float64　年齢に対する残高の比率
* 19) is_new_prospect：int64　previousが0かつpdaysが-1の完全新規フラグ。
* 20) day_month_interaction：int64　月と日の組み合わせによるボーナス・決算期特定

# ライブラリ

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# 学習モデル作成

In [2]:
df = pd.read_pickle('data.pkl')
df

,id,age,job,marital,education,default,balance,housing,loan,contact,...,month,campaign,pdays,previous,poutcome,y,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction
0,0,42,technician,married,secondary,no,7,no,no,cellular,...,aug,3,-1,0,unknown,0,0,0,1,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,...,jun,1,-1,0,unknown,0,0,0,1,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,...,may,2,-1,0,unknown,0,1,0,1,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,...,may,2,-1,0,unknown,0,1,0,1,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,...,feb,1,-1,0,unknown,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,749995,29,services,single,secondary,no,1282,no,yes,unknown,...,jul,2,-1,0,unknown,1,1,1,1,0
749996,749996,69,retired,divorced,tertiary,no,631,no,no,cellular,...,aug,1,-1,0,unknown,0,0,0,1,0
749997,749997,50,blue-collar,married,secondary,no,217,yes,no,cellular,...,apr,1,-1,0,unknown,0,1,0,1,0
749998,749998,32,technician,married,secondary,no,-274,no,no,cellular,...,aug,6,-1,0,unknown,0,0,0,1,0


In [3]:
y = df["y"]
x = df.drop(columns=["y", "id"])
x = pd.get_dummies(x, drop_first=True)


In [4]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42)

In [5]:
train_data = lgb.Dataset(x_train,label=y_train)
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'binary_logloss',
    'num_leaves': 31,
    'learning_rate': 0.05    
}

In [6]:
model = lgb.train(params, train_data,num_boost_round=100)

[LightGBM] [Info] Number of positive: 72283, number of negative: 527717
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014123 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 780
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120472 -> initscore=-1.987971
[LightGBM] [Info] Start training from score -1.987971


In [7]:
y_pred_proba = model.predict(x_test)
y_pred_proba

array([0.11348367, 0.65093543, 0.05068171, ..., 0.10937215, 0.12143439,
       0.29567518], shape=(150000,))

In [8]:
auc = roc_auc_score(y_test, y_pred_proba)
print("roc-auc : ",auc)

roc-auc :  0.8418812223868644


# テスト、提出

In [9]:
df_test = pd.read_pickle('test.pkl')
df_test

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction
0,750000,32,blue-collar,married,secondary,no,1397,yes,no,unknown,21,may,1,-1,0,unknown,1,1,1,0
1,750001,44,management,married,tertiary,no,23,yes,no,cellular,3,apr,2,-1,0,unknown,1,0,1,0
2,750002,36,self-employed,married,primary,no,46,yes,yes,cellular,13,may,2,-1,0,unknown,2,0,1,0
3,750003,58,blue-collar,married,secondary,no,-1380,yes,yes,unknown,29,may,1,-1,0,unknown,2,0,1,0
4,750004,28,technician,single,secondary,no,1950,yes,no,cellular,22,jul,1,-1,0,unknown,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,999995,43,management,married,tertiary,no,0,yes,no,cellular,18,nov,2,-1,0,unknown,1,0,1,0
249996,999996,40,services,married,unknown,no,522,yes,no,cellular,19,nov,1,189,1,failure,1,0,0,0
249997,999997,63,retired,married,primary,no,33,no,no,cellular,3,jul,1,92,8,success,0,0,0,0
249998,999998,50,blue-collar,married,primary,no,2629,yes,no,unknown,30,may,2,-1,0,unknown,1,1,1,0


In [10]:
submit_x = df_test.drop(columns=["id"])
submit_x = pd.get_dummies(submit_x, drop_first=True)
submit_x = submit_x.reindex(columns=x.columns, fill_value=0)

In [11]:
submit_x

,age,balance,day,campaign,pdays,previous,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,32,1397,21,1,-1,0,1,1,1,0,...,False,False,False,True,False,False,False,False,False,True
1,44,23,3,2,-1,0,1,0,1,0,...,False,False,False,False,False,False,False,False,False,True
2,36,46,13,2,-1,0,2,0,1,0,...,False,False,False,True,False,False,False,False,False,True
3,58,-1380,29,1,-1,0,2,0,1,0,...,False,False,False,True,False,False,False,False,False,True
4,28,1950,22,1,-1,0,1,1,1,0,...,True,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,43,0,18,2,-1,0,1,0,1,0,...,False,False,False,False,True,False,False,False,False,True
249996,40,522,19,1,189,1,1,0,0,0,...,False,False,False,False,True,False,False,False,False,False
249997,63,33,3,1,92,8,0,0,0,0,...,True,False,False,False,False,False,False,False,True,False
249998,50,2629,30,2,-1,0,1,1,1,0,...,False,False,False,True,False,False,False,False,False,True


In [12]:
test_pred = model.predict(submit_x)
test_pred

array([0.0546681 , 0.05170152, 0.06889162, ..., 0.81178263, 0.05000598,
       0.42755509], shape=(250000,))

In [13]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'y': test_pred
})
submission.to_csv('submission.csv', index=False)